
---


# **CS 5805, Homework 1, 100 points, 15% Credit**
##**Due before 11:59 PM Wednesday September 24, 2025**
---



**Instructions**:

1.   Honor code is enforced. This is an individual assignment. You should do your own work. Any evidence of copying will result in an immediate zero grade (0 point) and additional penalties/actions.
2.   Edits are only allowed at where 'TODO' tags exist. Edits made elsewhere will result in an immediate zero grade (0 point).
3.   Importing extra packages is forbidden. Any extra package import (including but not limited to numpy, scikit-learn, etc.) will result in an immediate zero grade (0 point).
4.   Please run each cell, including those that are collapsed, shown as `Show code`.
5.   Upon completion of this assignment, please download a '.ipynb' file through Taskbar > File > Download > .ipynb, then upload the file to Canvas.


### **1. Overview and Objective**
In this assignment, you will be implementing the C4.5 decision tree learning algorithm and running it on real emails for spam filtering.
In the class, we talked about how decision trees can be applied on data for features with categorical values.
In this assignment, we objective is to extend decision trees to handle data where features can be in continuous space.
In order to accomplish this objective, you will need to do some extra work in the part of data preprocessing and intelligently choosing "thresholds" to split data.

### **2. Data preprocessing [10 points]**

In real world problem settings, we need to split the dataset into multiple subsets of data first to get different groups.
To simplify, assume we only perform a binary splitting, where the dataset is splitted into two groups.
Instead of randomly splitting data into two groups, a better way might be to select a feature and use a threshold to determine whether a data point should be assigned to the first group (i.e., the left child node) or the second group (i.e. the right child node)



First, let's take a look at the format of the data.

In [46]:
#@title
class Point:
    def __str__(self):
        return "<{}:{}>".format(self.label, self.values)
    def __repr__(self):
        return "<{}:{}>".format(self.label, self.values)
    def __init__(self, label, values):
        self.label = label
        self.values = values

def get_label(s, labels):
    for l in labels:
        if l in s:
            return l
    raise Exception('Label not found', s)

def string_statistics(s):
    s = s.lower()
    values = [0]*32
    total = len(s)
    for c in s:
        n = ord(c)
        if 97 <= n <= 122:
            values[n-97] += 1
        elif c == '.':
            values[26] += 1
        elif c == ',':
            values[27] += 1
        elif c == '?':
            values[28] += 1
        elif c == '!':
            values[29] += 1
        elif c in '0123456789':
            values[30] += 1
        else:
            values[31] += 1
    for i in range(len(values)):
        values[i] /= float(total)
    return values


def get_spam_train_data(fn):
    with open(fn, 'rb') as f:
        train_data = pkl.load(f)
    return train_data

def get_spam_valid_data(fn):
    with open(fn, 'rb') as f:
        valid_data = pkl.load(f)
    return valid_data

def get_college_data():
    data = [
        Point('College', [24, 40000]),
        Point('No College', [53, 52000]),
        Point('No College', [23, 25000]),
        Point('College', [25, 77000]),
        Point('College', [32, 48000]),
        Point('College', [52, 110000]),
        Point('College', [22, 38000]),
        Point('No College', [43, 44000]),
        Point('No College', [52, 27000]),
        Point('College', [48, 65000])
    ]
    return data

In [47]:
data = get_college_data()
print(data)
print(data[0])
print(data[0].label)
print(data[0].values)

[<College:[24, 40000]>, <No College:[53, 52000]>, <No College:[23, 25000]>, <College:[25, 77000]>, <College:[32, 48000]>, <College:[52, 110000]>, <College:[22, 38000]>, <No College:[43, 44000]>, <No College:[52, 27000]>, <College:[48, 65000]>]
<College:[24, 40000]>
College
[24, 40000]



So `data` is just a list of `Point`s. Each `Point` has a label, in this case "College" or "No College", and each `Point` has a set of values.

Fill in the `split_data` function so that `left` contains all points whose value for the feature is less than the threshold and `right` contains all the other points.

In [48]:
#@title
class Point:
    def __str__(self):
        return "<{}:{}>".format(self.label, self.values)
    def __repr__(self):
        return "<{}:{}>".format(self.label, self.values)
    def __init__(self, label, values):
        self.label = label
        self.values = values

def get_label(s, labels):
    for l in labels:
        if l in s:
            return l
    raise Exception('Label not found', s)

def string_statistics(s):
    s = s.lower()
    values = [0]*32
    total = len(s)
    for c in s:
        n = ord(c)
        if 97 <= n <= 122:
            values[n-97] += 1
        elif c == '.':
            values[26] += 1
        elif c == ',':
            values[27] += 1
        elif c == '?':
            values[28] += 1
        elif c == '!':
            values[29] += 1
        elif c in '0123456789':
            values[30] += 1
        else:
            values[31] += 1
    for i in range(len(values)):
        values[i] /= float(total)
    return values


def get_spam_train_data(fn):
    with open(fn, 'rb') as f:
        train_data = pkl.load(f)
    return train_data

def get_spam_valid_data(fn):
    with open(fn, 'rb') as f:
        valid_data = pkl.load(f)
    return valid_data

def get_college_data():
    data = [
        Point('College', [24, 40000]),
        Point('No College', [53, 52000]),
        Point('No College', [23, 25000]),
        Point('College', [25, 77000]),
        Point('College', [32, 48000]),
        Point('College', [52, 110000]),
        Point('College', [22, 38000]),
        Point('No College', [43, 44000]),
        Point('No College', [52, 27000]),
        Point('College', [48, 65000])
    ]
    return data

In [49]:
#@title
from math import log
import pickle as pkl
class Tree:
    leaf = True
    prediction = None
    feature = None
    threshold = None
    left = None
    right = None

def predict(tree, point):
    if tree.leaf:
        return tree.prediction
    i = tree.feature
    if (point.values[i] < tree.threshold):
        return predict(tree.left, point)
    else:
        return predict(tree.right, point)

def most_likely_class(prediction):
    labels = list(prediction.keys())
    probs = list(prediction.values())
    return labels[probs.index(max(probs))]

def accuracy(data, predictions):
    total = 0
    correct = 0
    for i in range(len(data)):
        point = data[i]
        pred = predictions[i]
        total += 1
        guess = most_likely_class(pred)
        if guess == point.label:
            correct += 1
    return float(correct) / total

In [50]:
def split_data(data, feature, threshold):
    left = []
    right = []
    # TODO: split data into left and right by given feature.
    # left should contain points whose values are less than threshold
    # right should contain points with values greater than or equal to threshold
    for i in range(len(data)):
        point = data[i]
        if point.values[feature]<threshold:
            left.append(point)
        else:
            right.append(point)
    return (left, right)

In [51]:
def test_split():
    left, right = split_data(data, 0, 25)
    for point in left:
        assert point.values[0] < 25
    assert len(left) == 3

    for point in right:
        assert point.values[0] >= 25
    assert len(right) == 7
test_split()
print('Pass: Splitting Data [10 points]')

Pass: Splitting Data [10 points]


### **3. Calculating Entropy [10 points]**

The C4.5 algorithm finds partitions for the data that minimize entropy so we need to be able to calculate entropy. Entropy is defined as the negative sum across all events (in this case classes) of the probability of that event times the log probability of that event. More formally, the entropy of a random variable X is defined as: $H(X) = -\sum_{i} P(X=i) \log_2 P(X=i)$

To calculate the probabilities for each class in a given dataset, we first need to count the occurences of each class. Fill in `count_labels` to return a dictionary containing the number of times each label occurs in the data.

Next fill in `counts_to_entropy` to convert a dictionary of counts to the entropy of the data.

In [52]:
def count_labels(data):
    counts = {}
    # TODO: counts should count the labels in data
    # e.g. counts = {'spam': 10, 'ham': 4}
    for i in range(len(data)):
        label = data[i].label
        if label in counts:
            counts[label]+=1
        else:
            counts[label] = 1
    return counts

def counts_to_entropy(counts):
    entropy = 0.0
    total_count = sum(counts.values())
    # TODO: should convert a dictionary of counts into entropy
    for label, count in counts.items():
        px = count/total_count
        entropy -= px*log(px, 2)
    return entropy

In [53]:
#@title
def get_entropy(data):
    counts = count_labels(data)
    entropy = counts_to_entropy(counts)
    return entropy

In [54]:
def test_entropy():
    assert abs(get_entropy(data) - 0.9709505944546686) < 1e-7
test_entropy()
print('Pass: Calculating Entropy [10 points]')

Pass: Calculating Entropy [10 points]


### **4. Finding The Right Threshold [30 points]**

Given a dataset and some feature to split on, we need to be able to find the best split that gets us the most information gain. One way to do this is to look at every data point and try splitting on that data point's value for the feature. This method is shown in the function `find_best_threshold`.

While this will give the correct answer, it involves re-splitting the data numerous times and recalculating entropy from scratch which can be slow on large datasets. We need a faster way to find the best threshold.

An efficient way of doing this is to sort the dataset by the feature we are splitting on. Then we can go through the sorted data in order, moving data points from the `right` split to the `left` and keeping a rolling count of the probabilities of each label. This saves a lot of work when calculating information gain for each split. Implement this method in `find_best_threshold_fast`.

In [55]:
# This is a correct but inefficient way to find the best threshold to maximize
# information gain.
def find_best_threshold(data, feature):
    entropy = get_entropy(data)
    best_gain = 0
    best_threshold = None
    for point in data:
        left, right = split_data(data, feature, point.values[feature])
        curr = (get_entropy(left)*len(left) + get_entropy(right)*len(right))/len(data)
        gain = entropy - curr
        if gain > best_gain:
            best_gain = gain
            best_threshold = point.values[feature]
    return (best_gain, best_threshold)


def entropy_from_counts(counts):
    total = sum(counts.values())
    if total == 0:
        return 0
    return -sum((c / total) * log(c / total, 2) for c in counts.values() if c > 0)

def find_best_threshold_fast(data, feature):
    data = sorted(data, key=lambda x: x.values[feature])
    total_counts = count_labels(data)
    left_counts = {}
    right_counts = total_counts.copy()
    total_len = len(data)
    parent_entropy = entropy_from_counts(total_counts)

    best_gain = 0
    best_threshold = None

    for i in range(total_len - 1):
        point = data[i]
        label = point.label
        threshold = point.values[feature]

        left_entropy = entropy_from_counts(left_counts)
        right_entropy = entropy_from_counts(right_counts)
        weighted_entropy = (sum(left_counts.values())*left_entropy + sum(right_counts.values())*right_entropy)/total_len
        gain = parent_entropy - weighted_entropy

        if gain > best_gain:
            best_gain = gain
            best_threshold = threshold
        # this is moving
        left_counts[label] = left_counts.get(label, 0) + 1
        right_counts[label] -= 1
    return best_gain, best_threshold



In [56]:
def test_threshold():
    gain, thresh = find_best_threshold(data, 1)
    assert abs(gain - 0.321928094887) < 1e-7
    assert thresh == 38000

def test_threshold_fast():
    gain, thresh = find_best_threshold_fast(data, 1)
    assert abs(gain - 0.321928094887) < 1e-7
    assert thresh == 38000
test_threshold()
test_threshold_fast()
print('pass: Finding The Right Threshold [30 points]')

pass: Finding The Right Threshold [30 points]


### **5. Finding The Best Split [10 points]**

Now we can find the best split of the data over some features, but in order to run C4.5 we have to find the best split over all thresholds and all features. Fill in `find_best_split` to return the best feature and threshold that maximize information gain.

In [57]:
def find_best_split(data):
    if len(data) < 2:
        return None, None
    best_feature = None
    best_threshold = None
    best_gain = 0
    features = len(data[0].values)
    for i in range(features):
        gain, thresh = find_best_threshold_fast(data, i)
        if gain>best_gain:
            best_threshold = thresh
            best_feature = i
            best_gain = gain
        elif gain == best_gain and best_feature is not None and i>best_feature:
            best_threshold = thresh
            best_feature = i
    return (best_feature, best_threshold)

In [58]:
def test_best_split():
    feature, thresh = find_best_split(data)
    assert feature == 1
    assert thresh == 38000
    left, right = split_data(data, feature, thresh)
    feature, thresh = find_best_split(left)
    assert feature == None
    assert thresh == None
    feature, thresh = find_best_split(right)
    assert feature == 0
    assert thresh == 43
test_best_split()
print('pass: Finding The Best Split [10 points]')

pass: Finding The Best Split [10 points]


### **6. Finish C4.5 [20 points]**

Given some training data, we don't want to just split it one time. Rather, we want to keep going until we can't maximize information gain any further (or until some maximum tree depth is reached).

Fill in the `C4.5` function to do the following:

- If all the points have the same label, if no split can result in further maximization of information gain, or if the maximum tree depth has been reached, make a leaf with the data, recording the expected value of each label (this is provided in `make_leaf`.
- Otherwise split the data at the best threshold for the best feature. Make an internal (non-leaf) node for that feature and threshold.
- Create the left and right subtrees of the node by recursing on the two partitions of the data.

In [59]:
def make_leaf(data):
    tree = Tree()
    counts = count_labels(data)
    prediction = {}
    for label in counts:
        prediction[label] = float(counts[label])/len(data)
    tree.prediction = prediction
    return tree

def c45(data, max_levels):
    if max_levels <= 0 or len(data)<2:
        return make_leaf(data)
    # TODO: Construct a decision tree with the data and return it.
    # Your algorithm should return a leaf if the maximum level depth is reached
    # or if there is no split that results in further maximization of information gain,
    # otherwise it should greedily choose an feature and threshold to split on
    # and recurse on both partitions of the data.
    # You can create and return variables as needed.

    feature, thresh = find_best_split(data)
    if feature is None or thresh is None:
        return make_leaf(data)
    left, right = split_data(data, feature, thresh)
    if len(left)<2 or len(right)<2:
        return make_leaf(data)

    tree = Tree()
    tree.leaf = False
    tree.left = c45(left, max_levels-1)
    tree.right = c45(right, max_levels-1)
    tree.feature = feature
    tree.threshold = thresh
    return tree

### **7. Real world dataset [10 points]**

Once all the basic tests pass, it's time to run your algorithm on real data. We have provided two data files `train_data.pkl` and `valid_data.pkl` for training and validation respectively.

You will need to put the dataset in your Google drive, mount the dataset and grant read privilege to the notebook (see section **7 Loading Dataset** below).

This test will train your decision tree on about 20,000 emails labelled as spam and not spam ("ham"). We have pre-processed the data such that each feature represents a letter and the value of a feature is its number of occurrences.

Train your decision tree using the provided training dat and calculate your accuracy on the provided validation data. By default, it trains a single tree with a maximum depth of 4. This should take about 3 minutes and get you an accuracy of around 68% at classifying spam.

Tune your `submission` function to get your accuracy over 68% on the validation data. Describe what you did to increase your accuracy. How does changing the maximum depth of the tree affect your accuracy? Write a comment above your `submission` function.

Note that the real-world dataset is using 'Ham' and 'Spam' as labels, you may need to update your condition statements in previous cells to reflect the change of labels accordingly.

In [60]:
"""
This comment documents all my tries to take the accuracy as above as possible over 68%

 - Accuracy 78% (Increased the max_level to 10)
 - Accuracy 73% (As it turns out, increasing max_level to 30 isn't a great idea)
"""
def submission(train, test):
    # TODO: Once your tests pass, make your submission as good as you can!
    tree = c45(train, 10)
    predictions = []
    for point in test:
        predictions.append(predict(tree, point))
    return predictions

# This might be useful for debugging.
def print_tree(tree):
    if tree.leaf:
        print("Leaf", tree.prediction)
    else:
        print("Branch", tree.feature, tree.threshold)
        print_tree(tree.left)
        print_tree(tree.right)

### **8. Loading Dataset**
Upload the two provided data files `train_data.pkl` and `valid_data.pkl` to your Google Drive. Our default file path assumes your files are uploaded under directory `Colab Notebooks`. You may modify the file path within functions `get_spam_train_data` and `get_spam_valid_data` to reflect your file paths.

In [61]:
#Append the directory to your python path using sys
import sys
import os
prefix = '/content/drive/My Drive/'
# modify "customized_path_to_your_homework" here to where you uploaded your homework
customized_path_to_your_homework = 'Colab Notebooks/'
sys_path = prefix + customized_path_to_your_homework
sys.path.append(sys_path)
# print(sys.path)

fp_train = os.path.join(sys_path, 'train_data.pkl')
fp_valid = os.path.join(sys_path, 'valid_data.pkl')
print('Path to train_data.pkl: {}'.format(fp_train))
print('Path to valid_data.pkl: {}'.format(fp_valid))

Path to train_data.pkl: /content/drive/My Drive/Colab Notebooks/train_data.pkl
Path to valid_data.pkl: /content/drive/My Drive/Colab Notebooks/valid_data.pkl


In [62]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [63]:
def testsubmission(fp_train, fp_valid):
    # Please modify file path accordingly to load the datasets.
    train = get_spam_train_data(fp_train)
    valid = get_spam_valid_data(fp_valid)
    preds = submission(train, valid)
    acc = accuracy(valid, preds)
    print("Your current accuracy is:{}".format(acc))
    assert acc > .68
testsubmission(fp_train, fp_valid)
print('Pass: Real world dataset [10 points]')

Your current accuracy is:0.7496
Pass: Real world dataset [10 points]


### **9. Perfect Your Submission [10 points + EXTRA]**

Finalize your `submission` function to take a train and test data and return a list of predictions for the test data. You can stick with just training a single decision tree, you could try modifying the C4.5 algorithm, you could average predictions across multiple decision trees, or you could come up with something new!

After the homework deadline, we will run all of your submissions against an independent test dataset to evaluate your accuracy. Needless to say, the independent test data will be different from either the training or validation data so make sure you don't overfit your model too much. Extra points on the homework will be awarded to students with the highest performing algorithms.

You are not provided the data files `train_perfection_data.pkl` and `valid_perfection_data.pkl`, but we will run the following code and obtain your model accuracy on this withheld data. We will score your submission based on the performance of your algorithm against a predefined accuracy threshold we obtained during our in-house testing.

In [64]:
fp_train_perfection = os.path.join(sys_path, 'train_perfection_data.pkl')
fp_valid_perfection = os.path.join(sys_path, 'valid_perfection_data.pkl')
print('Path to train_perfection_data.pkl: {}'.format(fp_train_perfection))
print('Path to valid_perfection_data.pkl: {}'.format(fp_valid_perfection))

Path to train_perfection_data.pkl: /content/drive/My Drive/Colab Notebooks/train_perfection_data.pkl
Path to valid_perfection_data.pkl: /content/drive/My Drive/Colab Notebooks/valid_perfection_data.pkl


In [65]:
def testperfection(fp_train_perfection, fp_valid_perfection):
    # Please modify file path accordingly to load the datasets.
    train = get_spam_train_data(fp_train_perfection)
    valid = get_spam_valid_data(fp_valid_perfection)
    preds = submission(train, valid)
    acc = accuracy(valid, preds)
    print("Your current accuracy is:{}".format(acc))
testperfection(fp_train_perfection, fp_valid_perfection)
print('Pass: Extra dataset [10 points]')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/Colab Notebooks/train_perfection_data.pkl'

### **10. Acknowledgments**

We have adapted this assignment based on the programming portion of the homework assignment in the Machine Learning course previously offered at the University of Washington. The original UW assignment can be found here: https://courses.cs.washington.edu/courses/cse446/17sp/homework/2017SP_CSE446_HW1_new.pdf.

